# Visualize Biography Chunk Embeddings

Load embedded chunks from ChromaDB and plot with t-SNE, colored by biography section.

In [1]:
import sys
sys.path.insert(0, '..')

import chromadb
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

import config
from rag import db_load_embeds

In [2]:
chroma_client = chromadb.PersistentClient(config.CHROMA_PATH, config.CHROMA_CLIENT_SETTINGS)
chunks = db_load_embeds(chroma_client, config.CHROMA_COLLECTION_NAME)

sections = [c.metadata['section'] for c in chunks]
embeddings = np.array([c.embedding for c in chunks])

print(f'{len(chunks)} chunks, {len(set(sections))} sections, {embeddings.shape[1]}-d embeddings')

121 chunks, 15 sections, 3072-d embeddings


In [ ]:
# Assign colors by section, preserving insertion order
unique_sections = list(dict.fromkeys(sections))
section_mapping = {s: i for i, s in enumerate(unique_sections)}
colors = [section_mapping[s] for s in sections]

# Reduce to 2D with t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings) - 1))
reduced = tsne.fit_transform(embeddings)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(reduced[:, 0], reduced[:, 1], c=colors, cmap='tab20', s=80, alpha=0.7)

handles = [plt.Line2D([0], [0], marker='o', color='w',
           markerfacecolor=scatter.cmap(scatter.norm(section_mapping[s])),
           markersize=10, label=s) for s in unique_sections]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, title='Section')

ax.set_title('Biographical Embeddings by Section (t-SNE)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()